# 03. Indexing, Slicing & Selection Mechanics: Beginner Guide

### 📌 Overview
Master **03. Indexing, Slicing & Selection Mechanics: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Label-Based Selection**: Covers `df.loc[]` and `df.at[]`.
- **Position-Based Selection**: Covers `df.iloc[]` and `df.iat[]`.
- **Boolean Filtering**: Covers `df[condition]` and `df.query()`.
- **Conditional Replacement**: Covers `df.where()` and `df.mask()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount card_type  \
0       TX109326      C55082       M3549              607.78      Visa   
1       TX106376      C76616       M3068             1819.11      Visa   

  transaction_status device_type  account_age_months     transaction_date  \
0           Reversed      Mobile                   8  2026-02-17 08:28:57   
1            Pending         POS                  28          03-Jan-2025   

  region  is_fraud  
0  North         0  
1   West         1  


### 🔹 Label Selection with `df.loc[]`
- **What it does:** Selects transaction subsets by row index and column names.
- **Syntax:** `df.loc[0:5, ['transaction_id', 'transaction_amount', 'card_type']]`
- **Key Note:** `.loc[]` is label-based and includes the end label (`df.loc[0:3]` includes row label 3).

In [2]:
print('df.loc Selection (Rows 0-3):\n', df.loc[0:3, ['transaction_id', 'transaction_amount', 'card_type']])

df.loc Selection (Rows 0-3):
   transaction_id  transaction_amount card_type
0       TX109326              607.78      Visa
1       TX106376             1819.11      Visa
2       TX103301               64.08      Visa
3       TX110701             1025.73      Amex


### 🔹 Scalar Label Lookup with `df.at[]`
- **What it does:** Fast scalar extraction bypassing series overhead.
- **Syntax:** `df.at[0, 'transaction_amount']`
- **Operation:** `print('Fast Scalar (df.at[0, amount]):', df.at[0, 'transaction_amount'])`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [3]:
print('Fast Scalar (df.at[0, amount]):', df.at[0, 'transaction_amount'])

Fast Scalar (df.at[0, amount]): 607.78


### 🔹 Position Selection with `df.iloc[]`
- **What it does:** Extracts rows and columns strictly by 0-indexed integer coordinates.
- **Syntax:** `df.iloc[0:4, 0:3]`
- **Operation:** `print('df.iloc Selection (First 4 rows, first 3 cols):\n', df.iloc[0:4, 0:3])`
- **Key Note:** `.iloc[]` works exactly like Python list indexing: it is 0-indexed and excludes the stop boundary (`df.iloc[0:3]` gives rows 0, 1, and 2).

In [4]:
print('df.iloc Selection (First 4 rows, first 3 cols):\n', df.iloc[0:4, 0:3])

df.iloc Selection (First 4 rows, first 3 cols):
   transaction_id customer_id merchant_id
0       TX109326      C55082       M3549
1       TX106376      C76616       M3068
2       TX103301      C65296       M3352
3       TX110701      C42098       M5807


### 🔹 Scalar Position Lookup with `df.iat[]`
- **What it does:** Fast integer coordinate scalar lookup.
- **Syntax:** `df.iat[0, 3]`
- **Operation:** `print('Fast Position Scalar (df.iat[0, 3]):', df.iat[0, 3])`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [5]:
print('Fast Position Scalar (df.iat[0, 3]):', df.iat[0, 3])

Fast Position Scalar (df.iat[0, 3]): 607.78


### 🔹 Vectorized Boolean Mask Filtering
- **What it does:** Filters transactions by amount and card type using boolean vector masks.
- **Syntax:** `df[(df['transaction_amount'] > 500) & (df['is_fraud'] == 1)]`
- **Operation:** `fraud_high_val = df[(df['transaction_amount'] > 500.0) & (df['is_fraud'] == 1)]`
- **Key Note:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.

In [6]:
fraud_high_val = df[(df['transaction_amount'] > 500.0) & (df['is_fraud'] == 1)]
print(f'High-Value Fraud Transactions Found: {len(fraud_high_val)}')
print(fraud_high_val[['transaction_id', 'transaction_amount', 'card_type', 'region']].head(3))

High-Value Fraud Transactions Found: 1572
   transaction_id  transaction_amount   card_type region
1        TX106376             1819.11        Visa   West
18       TX111101             1998.80  MasterCard   West
19       TX107785             1806.44        Visa  North


### 🔹 Dynamic String Queries with `df.query()`
- **What it does:** Multi-threaded query filtering on transaction amounts.
- **Syntax:** `df.query('transaction_amount > 800 and is_fraud == 1')`
- **Operation:** `threshold = 800.0`
- **Key Note:** Pandas `.str` accessor methods automatically skip missing (`NaN`) values instead of raising an `AttributeError`.

In [7]:
threshold = 800.0
queried_fraud = df.query('transaction_amount > @threshold and is_fraud == 1')
print('Queried Fraud Count:', len(queried_fraud))

Queried Fraud Count: 1554


### 🔹 Conditional Retention with `df.where()`
- **What it does:** Preserves matching values and replaces non-matching with 0.0.
- **Syntax:** `df['transaction_amount'].where(df['transaction_amount'] > 100, other=0.0)`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [8]:
print('Where (> 100):\n', df['transaction_amount'].where(df['transaction_amount'] > 100, other=0.0).head())

Where (> 100):
 0     607.78
1    1819.11
2       0.00
3    1025.73
4     772.74
Name: transaction_amount, dtype: float64


### 🔹 Conditional Masking with `df.mask()`
- **What it does:** Replaces values exceeding 1000 with a cap value 1000.0.
- **Syntax:** `df['transaction_amount'].mask(df['transaction_amount'] > 1000, other=1000.0)`
- **Key Note:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.

In [9]:
print('Masked (> 1000 capped):\n', df['transaction_amount'].mask(df['transaction_amount'] > 1000, other=1000.0).head())

Masked (> 1000 capped):
 0     607.78
1    1000.00
2      64.08
3    1000.00
4     772.74
Name: transaction_amount, dtype: float64


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: In-Place Mutation vs SettingWithCopyWarning
- **Objective:** Q1: In-Place Mutation vs SettingWithCopyWarning
- **Approach:** Correctly assign risk flags to high-value transactions using `df.loc` to avoid SettingWithCopyWarning.
- **Syntax:** `df.loc[df['transaction_amount'] > 500, 'risk_tier'] = 'High'`

In [10]:
df_copy = df.head(10).copy()
df_copy.loc[df_copy['transaction_amount'] > 200, 'risk_tier'] = 'High'
print(df_copy[['transaction_id', 'transaction_amount', 'risk_tier']])

  transaction_id  transaction_amount risk_tier
0       TX109326              607.78      High
1       TX106376             1819.11      High
2       TX103301               64.08       NaN
3       TX110701             1025.73      High
4       TX103284              772.74      High
5       TX104210              198.47       NaN
6       TX106427              217.23      High
7       TX104105             1070.66      High
8       TX114893                 NaN       NaN
9       TX114584             1320.66      High
